# 00 — Explore Jeffi data on Razer

Quick first look at the local replica (`jeffi_replica` on localhost:5432).

**Refreshed nightly** by `scripts/replicate_jeffi.sh` from prod RDS.

Goal: get oriented — what tables exist, how big are they, and pull a few rows from the most useful ones for ML.

In [ ]:
import sys
print('Python:', sys.executable)
from jeffistores_labs.db import query, list_tables, describe

## Tables — sorted by row count

In [ ]:
tables = list_tables()
print(f'{len(tables)} tables in public schema, total ~{tables["rows_est"].sum():,} rows')
tables.head(25)

## Products — the table we'll fine-tune description generation on

In [ ]:
describe('products')

In [ ]:
products = query("""
    SELECT id, name, sku, price, mrp, brand_id, category_id,
           length(description) AS desc_len
      FROM products
     WHERE description IS NOT NULL
     ORDER BY id DESC
     LIMIT 10
""")
products

## Description length distribution

Useful for picking the right context window when we fine-tune.

In [ ]:
desc_stats = query("""
    SELECT
        COUNT(*) AS total,
        COUNT(description) AS with_desc,
        AVG(length(description))::int AS avg_chars,
        percentile_cont(0.5) WITHIN GROUP (ORDER BY length(description))::int AS p50,
        percentile_cont(0.95) WITHIN GROUP (ORDER BY length(description))::int AS p95,
        MAX(length(description)) AS max_chars
    FROM products
""")
desc_stats

## Orders — for demand forecasting / churn projects later

In [ ]:
orders_summary = query("""
    SELECT
        date_trunc('month', created_at)::date AS month,
        COUNT(*) AS order_count,
        SUM(total_amount)::numeric(12,2) AS total_revenue
    FROM orders
    WHERE created_at > now() - interval '12 months'
    GROUP BY 1
    ORDER BY 1
""")
orders_summary

## Search analytics — input for semantic search project (Month 2)

In [ ]:
# May or may not exist depending on schema — wrap in try
try:
    print(describe('search_logs'))
except Exception as e:
    print('search_logs not present yet:', e)